# QuantumDevices Model Demo

This notebook demonstrates the flat model API and a basic top-level `QuantumDevice`: ordered components, named operator-expression interactions, saved model and gate recipes, defaults, pretruncation, dressing, and effective components.

In [1]:
using Pkg
project_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : abspath(joinpath(pwd(), ".."))
Base.active_project() == joinpath(project_root, "Project.toml") || Pkg.activate(project_root)

using QuantumDevices
using QuantumToolbox
import LinearAlgebra: norm

## Components

In [2]:
transmon = Component(TransmonSpec(0.25, 20.0; ng = 0.0), :transmon)
mode1 = Component(ResonatorSpec(6.0; dimension = 4), :mode1)
mode2 = Component(ResonatorSpec(7.2; dimension = 3), :mode2)

display(transmon)
display(mode1)
display(mode2)

Component :transmon
  Spec: TransmonSpec
  Dimension: (Inf,)
  Parameters:
    EC = 0.25 [fixed, required] ∈ positive real
    EJ = 20.0 [fixed, required] ∈ positive real
    ng = 0.0 [fixed, required] ∈ real
  Operators:
    identity
    n
    tunneling
  Hamiltonian: 4 × EC × (n - ng × identity)² - EJ × tunneling / 2

Component :mode1
  Spec: ResonatorSpec
  Dimension: (4,)
  Parameters:
    frequency = 6.0 [fixed, required] ∈ positive real
  Operators:
    a
    adag
    n
    number
    … 2 more
  Hamiltonian: frequency × n

Component :mode2
  Spec: ResonatorSpec
  Dimension: (3,)
  Parameters:
    frequency = 7.2 [fixed, required] ∈ positive real
  Operators:
    a
    adag
    n
    number
    … 2 more
  Hamiltonian: frequency × n

## A Basic QuantumDevice and Saved ModelSpec

A `QuantumDevice` stores the symbolic components and named `OperatorExpr` interactions. `modelspec` resolves selected names in order, infers their parameters, and returns an unsaved recipe. Registering that recipe makes it available to `model(device, name)`. Component defaults still use short tuple paths such as `(:transmon, :ng)`.

In [3]:
g1 = DeviceParameter(:g1; default = 0.025)
g2 = DeviceParameter(:g2; default = 0.018)

transmon_mode1 = param(g1) * op(:transmon, :n) * op(:mode1, :q)
transmon_mode2 = param(g2) * op(:transmon, :n) * op(:mode2, :q)

multimode_device = QuantumDevice(
    :multimode_device;
    components = (transmon, mode1, mode2),
    interactions = (
        :transmon_mode1 => transmon_mode1,
        :transmon_mode2 => transmon_mode2,
    ),
)

multimode_spec = modelspec(
    multimode_device,
    :multimode_cavity,
    (:transmon, :mode1, :mode2),
    (:transmon_mode1, :transmon_mode2);
    dims = Dict(:transmon => 4, :mode1 => 4, :mode2 => 3),
    pretruncation_dims = Dict(:transmon => 15),
    defaults = Dict((:transmon, :ng) => 0.05),
    dressingspec = DressingSpec(steps = 8, minimum_overlap = 0),
)

register!(multimode_device, multimode_spec)
multimode = model(multimode_device, :multimode_cavity)

QuantumDeviceModel :multimode_cavity
  Hamiltonian: 48×48
  Operators: 16
  States: 48
  Energies: 48
  Minimum dressing overlap: 0.9818197145611466

In [4]:
Dict(:test => 1)

Dict{Symbol, Int64} with 1 entry:
  :test => 1

In [5]:
reference_label = (0, 0, 0)

(
    component_order = keys(multimode.spec.subsystems),
    dimensions = multimode.spec.dimension,
    hamiltonian_size = size(multimode.hamiltonian),
    dressed_state_count = length(multimode.states),
    reference_energy = multimode.energies[reference_label],
    minimum_overlap = minimum(multimode.dressing_res.overlaps),
    parameter_paths = collect(keys(multimode.spec.parameters)),
    device = multimode_device,
)

(component_order = (:transmon, :mode1, :mode2), dimensions = Dimension((4, 4, 3)), hamiltonian_size = (48, 48), dressed_state_count = 48, reference_energy = -16.90165799539367, minimum_overlap = 0.9818197145611466, parameter_paths = ParamPath[transmon/EC, g2, mode2/frequency, mode1/frequency, transmon/ng, g1, transmon/EJ], device = QuantumDevice(:multimode_device; components=[mode1, mode2, transmon], interactions=[transmon_mode1, transmon_mode2], models=[multimode_cavity], gates=[]))

## Reusing a Model as a Component

Nesting stays explicit: build a model, freeze its lowest dressed levels into a component, then pass that component to another spec.

In [6]:
multimode_component = component(multimode; name = :multimode, dimension = 8)
projected_mode_operator = numerical(multimode_component, (:mode1, :q))

(
    component = multimode_component,
    operator_size = size(projected_mode_operator),
)

(component = Component(:multimode, GenericSpec; 0 parameters, 18 operators), operator_size = (8, 8))

## Flux-Tunable Coupler

The parameter registry combines short component paths with explicitly supplied interaction parameters. Non-fixed component parameters are directly discoverable as controls.

In [7]:
qubit1 = Component(FluxTunableTransmonSpec(0.25, 20.0), :qubit1)
coupler = Component(FluxTunableTransmonSpec(0.25, 21.0), :coupler)
qubit2 = Component(FluxTunableTransmonSpec(0.25, 22.0), :qubit2)

g1c = DeviceParameter(:g1c; default = 0.025)
g2c = DeviceParameter(:g2c; default = 0.028)
g12 = DeviceParameter(:g12; default = 0.001)

tunable_spec = ModelSpec(
    :tunable_coupler,
    [qubit1, coupler, qubit2],
    Dict(:qubit1 => 3, :coupler => 3, :qubit2 => 3);
    interactions = (
        param(g1c) * op(:qubit1, :n) * op(:coupler, :n),
        param(g2c) * op(:qubit2, :n) * op(:coupler, :n),
        param(g12) * op(:qubit1, :n) * op(:qubit2, :n),
    ),
    parameters = (g1c, g2c, g12),
    pretruncation_dims = Dict(:qubit1 => 11, :coupler => 11, :qubit2 => 11),
    defaults = Dict((:coupler, :flux) => 0.1),
    dressingspec = DressingSpec(steps = 8, minimum_overlap = 0),
)

tunable_model = model(tunable_spec)
controls = Dict(path => parameter for (path, parameter) in tunable_spec.parameters if !parameter.fixed)

(model = tunable_model, controls = controls)

(model = QuantumDeviceModel(:tunable_coupler; dimension=27, 12 operators), controls = Dict{ParamPath, DeviceParameter}(coupler/flux => Parameter(flux=0.0), qubit1/flux => Parameter(flux=0.0), qubit2/flux => Parameter(flux=0.0)))

## Gate Overlay

`GateSpec` stores only symbolic expressions and trajectories. After registering it on a device, `numerical(device, gate_name)` builds the complete time-dependent Hamiltonian through the gate's saved `ModelSpec`.

In [8]:
variable_qubit = Component(QubitSpec(5.0), :q)
variable_device = QuantumDevice(:variable_qubit; components = (variable_qubit,))
variable_spec = modelspec(
    variable_device,
    :variable_frequency_qubit,
    (:q,);
    dressingspec = DressingSpec(minimum_overlap = 0),
)
register!(variable_device, variable_spec)
variable_model = model(variable_device, :variable_frequency_qubit)

duration = 1.0
amplitude = 0.2
x_drive = DeviceParameter(
    :x_drive;
    fixed = false,
    default = t -> amplitude * sin(pi * t / duration),
)
x_gate = GateSpec(
    :x_gate,
    variable_spec;
    duration = duration,
    hamiltonian = param(x_drive) * op(:q, :x),
    parameters = (x_drive,),
)
register!(variable_device, x_gate)
x_gate_hamiltonian = numerical(variable_device, :x_gate)

driven = x_gate_hamiltonian(nothing, duration / 2)
expected = variable_model.hamiltonian + amplitude * variable_model.operators[(:q, :x)]

(device = variable_device, gate = x_gate_hamiltonian, numerical_check = norm(driven - expected))

(device = QuantumDevice(:variable_qubit; components=[q], interactions=[], models=[variable_frequency_qubit], gates=[x_gate]), gate = 
Quantum Object Evo.:   type=Operator()   dims=([2], [2])   size=(2, 2)   ishermitian=true   isconstant=false
(MatrixOperator(2 × 2) + ScalarOperator(0.2 + 0.0im) * MatrixOperator(2 × 2)), numerical_check = 0.0)